# 0. Packages

In [1]:
import pandas as pd
import os

# 1. Relevant scraping information and data

**NOTE**: *the scraping has been undertaken considering that we will do a difference-in-differences analysis of the price impact of the Mobile World Congress in Barcelona (which lasts from the 3rd of March to the 6th of March in 2025).

About the scraping:
1. The scraping of the results and descriptions has been done on the 29th January 2025. 
2. The dates comprised where accommodation information has been extracted are:
    - Start date: February 2025.
    - End date: June 2025.
3. For each month, information has been scraped for the 1st to the 8th, for the 9th to the 16th and for the 17th to the 24th.
4. If you want to try out that the `booking_scraping.py` works, the range of months between the start and the end must include, at least, a month in the future. E.g., if we are in February 2025, you must include at least March 2025 as the end month.
5. The duration of the scraping depends (mainly) on your internet connection. If your connection is not very stable, increase the parameter `time_sleep` in the script `booking_scraping.py`.
6. With a `time_sleep` of 2 seconds, the setting described above, 5452 distinct accommodations overall and a relatively stable connection the scraping has lasted for, approximately, 5 hours (1 hours 30 minutes the extraction of the general accommodation information and 3 hours 30 minutes the extraction of the descriptions of 5452 accommodations through requests).

About where to locate the data and how to extract useful information from it:
1. All the data that has been extracted is located in the `files` folder within the repo.
2. Inside the `files` folder, the `general_results` folder contains .csv where each file is general accommodation data for one place and one month (the range extracted is indicated in the file itself).
    - Features included: name of the accommodation, price, rating, number of reviews, quality of the hotel or accommodation, type of quality (stars or "squares"), neighborhood of the accommodation and the short description in the results list. The neighborhood was included considering the possibility that accommodation names were not unique, but it seems that they have to be unique according to Booking policies.
    - Data is not in the same file in order to avoid data loss while doing the scraping (if we saved all of the data only in the end, the progress would have been lost).
    - How to analyze it: I would recommend joining the whole .csv files into one or two separate data frames (depending on whether you want to include Barcelona and Madrid in the same data frame, which should not be a problem given that the neighborhood column includes the city name), where in both cases you should tag (i.e., add an additional column) the week for which the information has been extracted (i.e., 2025-05-01_2025-05-08, in yyyy-mm-dd format). But whatever you do, don't remove the original data.
    - This data extraction has been done with Selenium.
3. On the other hand, also inside the `files` folder, the `accommodation_descriptions` folder contains the file the file `descriptions_5452.csv` with the descriptions of all of the unique accommodations for the whole period that has been scraped.
    - Features included: name of the accommodation, neighborhood, URL (extracted when scraping the general results, but it is not useful anymore if we have the description - you can drop it in the analysis) and the description (in English! Can also be done in any other language, but this was the preferred choice for obvious reasons). 
    - When doing the scraping, it progressively saved the data as it increased the number of descriptions scraped. But in the end I have only kept the file with all the descriptions, `descriptions_5452.csv`.
    - For matching the descriptions with the general information of the accommodations, you can do it with the name of the accommodation. If the name of the accommodation was not enough to uniquely identify an accommodation, you can also include the neighborhood. 

# 2. Checking the `general_results` files
**WARNING**: don't concatenate the data the way I'm doing it below, where I'm not identifying the accommodation info by date to which the information corresponds to. I'm just doing it this way because I just want to check if everything was saved correctly (and to see if there are any error patterns).

In [2]:
results_data_path = "files/general_results/"

# Step 1: import all URL .csv into dataframes in a loop
dataframes = []
# os.listdir(folder_path) lists all files in the specified folder
for file in os.listdir(results_data_path):
    if file.endswith(".csv"):
        file_path = os.path.join(results_data_path, file)
        df_url = pd.read_csv(file_path)
        dataframes.append(df_url)

# Step 2: concatenate all dataframes along the rows (indexes, axis = 0)
df_results_combined = pd.concat(dataframes, axis=0, ignore_index=True)

In [3]:
df_results_combined

,hotel_name,price_euros,rating,num_reviews,quality_over_5,quality_type,neighborhood,description
0,Be Mate Paseo de Gracia,2517.0,8.9,1206.0,4.0,squares,"Gràcia, Barcelona",Penthouse Apartment\r\nEntire apartment • 2 be...
1,Tembo Barcelona,2598.0,8.8,2660.0,4.0,stars,Barcelona,One-Bedroom Premium Apartment\r\nEntire apartm...
2,Unite Hostel Barcelona,1079.0,8.0,8415.0,NaN,NaN,"Sant Martí, Barcelona",Family Room with Private Bathroom\r\n8 bunk beds
3,Catalonia Sagrada Familia,1207.0,8.4,9406.0,3.0,stars,"Sant Martí, Barcelona",Double or Twin Room\r\nBeds: 1 double or 2 twi...
4,NH Collection Barcelona Gran Hotel Calderon,2178.0,8.2,2920.0,5.0,stars,"Eixample, Barcelona",Superior Double or Twin Room\r\nBeds: 1 double...
...,...,...,...,...,...,...,...,...
27005,Beautiful flat in Lavapiés-Embajadores,1232.0,8.4,8.0,3.0,squares,"Madrid City Center, Madrid","One-Bedroom Apartment\r\n2 beds (1 full, 1 sof..."
27006,Apartamentos Duque Ventas,1574.0,9.5,57.0,4.0,squares,"Salamanca, Madrid",Two-Bedroom Apartment\r\nEntire apartment • 2 ...
27007,Style Suites by Olala Homes,1626.0,8.1,263.0,3.0,squares,"Puente de Vallecas, Madrid",Double Room\r\n1 full bed\r\nOnly 4 rooms left...
27008,Hostal Matheu,1220.0,7.6,667.0,3.0,squares,"Madrid City Center, Madrid",Double or Twin Room\r\nMultiple bed types\r\nO...


In [5]:
df_results_combined.info()
print("Current Working Directory:", os.getcwd())
df_results_combined.to_csv("df_results_combined.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27010 entries, 0 to 27009
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   hotel_name      27010 non-null  object 
 1   price_euros     27010 non-null  float64
 2   rating          26447 non-null  float64
 3   num_reviews     26447 non-null  float64
 4   quality_over_5  23308 non-null  float64
 5   quality_type    23308 non-null  object 
 6   neighborhood    27010 non-null  object 
 7   description     27009 non-null  object 
dtypes: float64(4), object(4)
memory usage: 1.6+ MB
Current Working Directory: d:\BSE Semester 2\Introduction to Text Mining and NLP\Assignments and Exercises\Assign 1\ITM_hw1


# 3. Checking the `accommodation_descriptions`

In [4]:
df_descriptions = pd.read_csv('files/accommodation_descriptions/descriptions_5452.csv')
df_descriptions

,hotel_name,neighborhood,url,description
0,The Oliver Apartamentos Aravaca,"Moncloa-Aravaca, Madrid",https://www.booking.com/hotel/es/the-oliver-ap...,"Offering garden views and a garden, The Oliver..."
1,LUXURY DESIGN Quevedo-Neptuno,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/luxury-design...,Located within less than 1 km of Reina Sofia M...
2,Live it Madrid Gran Vía,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/live-it-gran-...,"Offering city views, Live it Madrid Gran Vía i..."
3,Ibis Budget Madrid Centro Las Ventas,"Ciudad Lineal, Madrid",https://www.booking.com/hotel/es/ibis-budget-m...,"Featuring free Wi-Fi, Ibis Budget Madrid Centr..."
4,Apartamentos Juan Bravo,"Salamanca, Madrid",https://www.booking.com/hotel/es/apartamentos-...,The Juan Bravo Apartments are situated in the ...
...,...,...,...,...
5447,Atico Carmen CALLAO,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/atico-carmen-...,"Right in the centre of Madrid, set within a sh..."
5448,Loft con Terraza Madrid Salitre,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/loft-centro-m...,"Right in the centre of Madrid, situated within..."
5449,MyHouseSpain - Sensacional piso en Malasaña,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/myhousemadrid...,"Set in Madrid, less than 1 km from Gran Via Me..."
5450,"Habitación privada, luminosa y bien comunicada","Latina, Madrid",https://www.booking.com/hotel/es/habitacion-pr...,"Habitación privada, luminosa y bien comunicada..."


In [3]:
df_descriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6037 entries, 0 to 6036
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   hotel_name    6037 non-null   object
 1   neighborhood  6037 non-null   object
 2   url           6037 non-null   object
 3   description   6037 non-null   object
dtypes: object(4)
memory usage: 188.8+ KB


There are no missing values!! Also, from an exploratory analysis of the data frame it seems that all of the descriptions are correctly matched to the hotel.